# Data Exploration

## Initial check

In [35]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [36]:
df = pd.read_csv(
    "../data/raw/sfpd_bike_thefts.csv",
    parse_dates=["incident_datetime", "incident_date", "report_datetime"],
)

In [37]:
df.shape
df.head()

,row_id,incident_datetime,incident_date,incident_time,incident_year,incident_day_of_week,report_datetime,incident_id,incident_number,cad_number,report_type_code,report_type_description,incident_code,incident_category,incident_subcategory,incident_description,resolution,intersection,cnn,police_district,analysis_neighborhood,supervisor_district,supervisor_district_2012,latitude,longitude,data_as_of,data_loaded_at,filed_online
0,112231406314,2022-02-12 00:05:00,2022-02-12,00:05,2022,Saturday,2022-02-16 07:05:00,1122314,220106343,220470407.0,II,Initial,6314,Larceny Theft,Larceny Theft - Bicycle,"Theft, Bicycle, >$950",Open or Active,BAY ST \ POWELL ST,25461000.0,Central,North Beach,3.0,3.0,37.805824,-122.411949,2025-06-12T10:07:02.000,2025-06-13T09:52:57.000,NaN
1,112232706313,2022-01-10 10:00:00,2022-01-10,10:00,2022,Monday,2022-02-16 12:54:00,1122327,220106854,220471049.0,II,Initial,6313,Larceny Theft,Larceny Theft - Bicycle,"Theft, Bicycle, $200-$950",Open or Active,FILLMORE ST \ HAYES ST,25953000.0,Northern,Hayes Valley,5.0,5.0,37.775833,-122.431198,2025-06-12T10:07:02.000,2025-06-13T09:52:57.000,NaN
2,112257906314,2022-01-13 16:00:00,2022-01-13,16:00,2022,Thursday,2022-01-13 23:47:00,1122579,226023169,NaN,II,Coplogic Initial,6314,Larceny Theft,Larceny Theft - Bicycle,"Theft, Bicycle, >$950",Open or Active,16TH ST \ OWENS ST,23784000.0,Southern,Mission Bay,6.0,6.0,37.766693,-122.392662,2025-06-12T10:07:02.000,2025-06-13T09:52:57.000,True
3,112274506314,2022-02-16 15:00:00,2022-02-16,15:00,2022,Wednesday,2022-02-17 15:20:00,1122745,220109886,220482096.0,II,Initial,6314,Larceny Theft,Larceny Theft - Bicycle,"Theft, Bicycle, >$950",Open or Active,MISSION ST \ PRECITA AVE,21340000.0,Ingleside,Bernal Heights,9.0,9.0,37.746777,-122.419121,2025-06-12T10:07:02.000,2025-06-13T09:52:57.000,NaN
4,112299706314,2021-12-18 00:00:00,2021-12-18,00:00,2021,Saturday,2022-01-15 12:20:00,1122997,226024236,NaN,II,Coplogic Initial,6314,Larceny Theft,Larceny Theft - Bicycle,"Theft, Bicycle, >$950",Open or Active,LAFAYETTE ST \ MISSION ST,24364000.0,Southern,Mission,6.0,6.0,37.773476,-122.418030,2025-06-12T10:07:02.000,2025-06-13T09:52:57.000,True


We have a daaset of stolen bike reports, which contain various incident ids, timestamp, location, and whether an online report was filed.  

In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3882 entries, 0 to 3881
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   row_id                    3882 non-null   int64         
 1   incident_datetime         3882 non-null   datetime64[us]
 2   incident_date             3882 non-null   datetime64[us]
 3   incident_time             3882 non-null   str           
 4   incident_year             3882 non-null   int64         
 5   incident_day_of_week      3882 non-null   str           
 6   report_datetime           3882 non-null   datetime64[us]
 7   incident_id               3882 non-null   int64         
 8   incident_number           3882 non-null   int64         
 9   cad_number                3141 non-null   float64       
 10  report_type_code          3882 non-null   str           
 11  report_type_description   3882 non-null   str           
 12  incident_code             3882 

The data seems well populated, with null filed_online mapping to False.  116 are missing location data.

In [41]:
display(df.head(4).T)
len(df), df["row_id"].nunique(), df["incident_id"].nunique(), df[
    "incident_number"
].nunique(), df["incident_code"].nunique()

,0,1,2,3
row_id,112231406314,112232706313,112257906314,112274506314
incident_datetime,2022-02-12 00:05:00,2022-01-10 10:00:00,2022-01-13 16:00:00,2022-02-16 15:00:00
incident_date,2022-02-12 00:00:00,2022-01-10 00:00:00,2022-01-13 00:00:00,2022-02-16 00:00:00
incident_time,00:05,10:00,16:00,15:00
incident_year,2022,2022,2022,2022
incident_day_of_week,Saturday,Monday,Thursday,Wednesday
report_datetime,2022-02-16 07:05:00,2022-02-16 12:54:00,2022-01-13 23:47:00,2022-02-17 15:20:00
incident_id,1122314,1122327,1122579,1122745
incident_number,220106343,220106854,226023169,220109886
cad_number,220470407.0,220471049.0,NaN,220482096.0


(3882, 3882, 3878, 3689, 6)

Noting here that we have 3882 rows, 3878 incident ids, and 3689 incident numbers.  Lets reconcile how to evaluate a single theft, as rows likely overcounts since each row is a report.

In [40]:
cols = [
    "incident_number",
    "incident_id",
    "report_type_description",
    "incident_datetime",
    "report_datetime",
    "incident_description",
    "resolution",
]
dups = df[df.duplicated("incident_number", keep=False)].sort_values(
    ["incident_number", "report_datetime"]
)
dups[cols].head(12)

,incident_number,incident_id,report_type_description,incident_datetime,report_datetime,incident_description,resolution
890,180047055,625319,Initial,2018-01-17 17:42:00,2018-01-18 14:24:00,"Theft, Bicycle, >$950",Open or Active
893,180047055,625622,Initial Supplement,2018-01-17 17:42:00,2018-01-19 11:50:00,"Theft, Bicycle, >$950",Open or Active
901,180074412,629173,Initial,2018-01-26 18:30:00,2018-01-28 12:09:00,"Theft, Bicycle, $200-$950",Open or Active
902,180074412,630034,Initial Supplement,2018-01-26 18:30:00,2018-01-30 21:00:00,"Theft, Bicycle, $200-$950",Open or Active
908,180088316,631159,Initial,2018-01-27 18:54:00,2018-02-02 11:46:00,"Theft, Bicycle, $200-$950",Open or Active
935,180088316,635942,Initial Supplement,2018-02-15 08:27:00,2018-02-15 08:27:00,"Theft, Bicycle, $200-$950",Cite or Arrest Adult
927,180113123,634441,Initial,2018-02-11 09:00:00,2018-02-11 15:35:00,"Theft, Bicycle, >$950",Open or Active
947,180113123,637708,Initial Supplement,2018-02-13 17:57:00,2018-02-13 17:57:00,"Theft, Bicycle, >$950",Open or Active
931,180113123,635210,Initial Supplement,2018-02-11 09:00:00,2018-02-13 20:00:00,"Theft, Bicycle, >$950",Open or Active
944,180132101,637129,Initial,2018-02-18 17:20:00,2018-02-18 18:04:00,"Theft, Bicycle, $50-$200",Open or Active


Ok so after looking at website and this data, I believe each theft event has an incident_number.  Each report on that theft has an incident_id, and each bike in that theft potentially generates a new column based on its value/incident_code.

## Cleaning decisions

**Unit of analysis:** one row per `incident_number` — a theft event (could be multiple bikes; no way to know from the data).

**Columns to keep, and how to dedupe**

We will group by incident_number, order by report_datetime descending.  note that first means first non-null, so we can pull location data from later reports if initially missing.

| Column | Rule |
|---|---|
| `incident_number` | **key** |
| `report_datetime` | first |
| `incident_datetime` | first |
| `incident_description` | maximum value reported |
| `resolution` | last |
| `intersection` | first |
| `analysis_neighborhood` | first |
| `police_district` | first |
| `latitude` | first |
| `longitude` | first |
| `filed_online` | any True → True, otherwise False |

**Columns to create**

| Column | Type |
| --- | --- |
| `attempted_theft` | bool |
| `has_location` | bool |
| `n_reports` | int |
| `report_lag_days` | int |


